> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# IFR2 (RSI 2) de compra: backtest and optimization

One ticker, one strategy, from candles to charts. The rules live in `docs/brief.md`, section 7 (IFR2 subsection); this notebook only sets parameters and calls `b3core` (engine, metrics, score, optimizer). It holds no trading logic.

**Before trusting any number here:**
- **Survivorship bias.** The ticker you pick is one that still trades today. Strategies look better on survivors than they would have on the list of stocks you could have chosen years ago.
- **One ticker, many trials.** The best of hundreds of parameter combinations is partly luck. Always read the trial count next to the best result, and the out-of-sample column next to the in-sample one.
- **Daily candles only.** The spec is daily. Other timeframes run but raise a warning and are outside the spec.
- **Prices are raw, not adjusted.** Dividends and splits are ignored (`auto_adjust=False`), which understates total return and can create fake gaps around splits.
- **Costs.** The preset charges 0.03% per side (B3 fees, unverified, `docs/brief.md` section 10). Slippage and taxes are not modelled.
- **Entry at the signal close.** IFR2 buys at the close of the candle whose RSI triggers. In live trading that means placing the order shortly before the close, a named exception to the general rule that a signal acts on the next candle.
- The sources behind the strategy are research notes whose numbers are not verified.

In [ ]:
from datetime import UTC, datetime

import pandas as pd

from b3core.data import Timeframe
from b3core.ifr2 import Ifr2Config, run_ifr2
from b3core.score import score_v1
from b3report.charts import ifr2_equity_figure, monthly_returns_figure

pd.options.display.float_format = "{:,.4f}".format

## 1. Parameters

Everything the run depends on is a plain variable here. `Ifr2Config.preset` is the defaults plus the 0.03% fee per side. Sizing is by ATR: the quantity is `max_risk_brl / (atr_mult * ATR)`, rounded down to the lot and capped by the capital. There is no stop loss, so a single trade can lose more than `max_risk_brl`.

In [ ]:
SYMBOL = "LREN3"  # B3 ticker; Yahoo Finance adds the ".SA" suffix
START = datetime(2012, 1, 1, tzinfo=UTC)
END = datetime(2027, 1, 1, tzinfo=UTC)  # exclusive

CONFIG = Ifr2Config.preset(
    rsi_period=2,
    rsi_entry=30,  # grid 5..40
    exit_mode="target",  # "target", "rsi" or "target_or_rsi"
    rsi_exit=70,  # only used by the RSI exit modes
    target_lookback=2,  # exit above the highest high of the last N candles; grid 2..10
    max_days=None,  # time stop in candles; None = off
    allow_same_day_reentry=False,
    atr_period=5,
    atr_mult=2,
    max_risk_brl=2_000,
    capital_brl=100_000,
    lot_size=1,
)

## 2. Load candles

Daily candles from Yahoo Finance through `yfinance`, with raw (unadjusted) prices. This is the only cell that touches the network. It needs the `ifr2` dependency group: start Jupyter with `uv run --group ifr2 jupyter lab` from `backend/`. The data is vendor data: do not commit it.

In [ ]:
import yfinance as yf

raw = yf.download(
    f"{SYMBOL}.SA", start=START.date(), end=END.date(), auto_adjust=False, progress=False
)
raw.columns = raw.columns.get_level_values(0)  # newer yfinance returns two header levels
raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
candles = pd.DataFrame(
    {
        "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
        "open": raw["Open"].to_numpy(),
        "high": raw["High"].to_numpy(),
        "low": raw["Low"].to_numpy(),
        "close": raw["Close"].to_numpy(),
        "volume": raw["Volume"].to_numpy(dtype=float),
    }
).reset_index(drop=True)
print(f"{SYMBOL} {Timeframe.D1.value}: {len(candles)} candles, {candles.ts.iloc[0].date()} to {candles.ts.iloc[-1].date()}")
candles.head()

## 3. Check the engine against the worked example

The seven candles of brief section 7, with `atr_period` 2 and a max risk of R$ 100. Worked out by hand: RSI(2) first appears on bar 2 and falls to 17.86 on bar 4, below 30, so the signal is on bar 4 and the buy is at its close, 9.50. ATR(2) there is 0.5375, so the risk distance is `2 * 0.5375 = 1.075` and the quantity is `floor(100 / 1.075) = 93`. Bar 5 does not reach the target (10.30); bar 6 reaches the target, 10.00, so **+R$ 46.50** after 2 candles.

In [ ]:
example = pd.DataFrame(
    {
        "ts": pd.date_range("2024-01-02", periods=7, freq="B", tz="UTC"),
        "open": [10.00, 10.00, 10.50, 10.20, 9.80, 9.60, 9.90],
        "high": [10.10, 10.60, 10.60, 10.30, 9.90, 10.00, 10.10],
        "low": [9.90, 10.00, 10.10, 9.70, 9.40, 9.50, 9.80],
        "close": [10.00, 10.50, 10.20, 9.80, 9.50, 9.90, 10.00],
        "volume": 1000.0,
    }
)
EXAMPLE_CONFIG = Ifr2Config(
    atr_period=2, max_risk_brl=100, capital_brl=10_000, lot_size=1
)  # the same defaults as above: RSI(2) <= 30, target on the last 2 highs
(trade,) = run_ifr2(example, EXAMPLE_CONFIG).trades
print(f"entry {trade.entry:.2f}  exit {trade.exit:.2f} ({trade.exit_reason})  qty {trade.qty}  pnl R$ {trade.pnl:.2f}  days {trade.days}")
assert round(trade.pnl, 2) == 46.50 and trade.qty == 93, "worked example failed"

## 4. Backtest

One run of `CONFIG` on the whole history. Two drawdowns are reported and they differ on purpose: `max_drawdown` is measured on closed trades only (the balance, over the initial capital, brief section 6), while `max_equity_drawdown` marks the open position at each close and is measured from the equity peak. Calmar is the annualised return over the equity drawdown. The trade that is still open on the last candle is not counted.

In [ ]:
result = run_ifr2(candles, CONFIG)
metrics = result.metrics
summary = pd.Series(metrics.to_dict(), name=f"{SYMBOL} IFR2")
summary["calmar"] = metrics.calmar
summary["score_v1"] = score_v1(metrics)
summary.to_frame()

`score_v1` (`docs/score.md`) combines profit factor, equity drawdown and expectancy, and shrinks the result when there are few trades. It describes this one period only: out-of-sample is a separate flag in section 5.

In [ ]:
trades = pd.DataFrame([vars(t) for t in result.trades])
trades.head()

In [ ]:
trades.groupby("exit_reason").agg(trades=("pnl", "size"), total_pnl=("pnl", "sum"), avg_days=("days", "mean"))

In [ ]:
display(ifr2_equity_figure(result, candles))
monthly_returns_figure(result)